In [1]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt

# --------- Config (edit these) ---------
h5_path = '../../../../project/lofarsw/Data/Dynspec/unlabelled_dataset_oriented_unf.h5'
target_size = (64, 64)     # resize for display (block-avg)
cmap = "inferno"
cols = 10                  # images per row in the grid
n_samples = 50             # how many images to show in total
random_pick = True         # True = random selection, False = first n_samples
seed = 42                  # used if random_pick=True
use_percentile_contrast = True  # per-image [p1, p99] contrast
indices_override = []      # e.g. [0, 5, 123, 999]; non-empty overrides random/first

# --------- Helpers ---------
def downsample(img, target_size):
    """Block-avg downsampling to target_size."""
    H, W = img.shape
    h_new, w_new = target_size
    h_stride = max(1, H // h_new)
    w_stride = max(1, W // w_new)
    h_used = (H // h_stride) * h_stride
    w_used = (W // w_stride) * w_stride
    return img[:h_used, :w_used].reshape(h_used//h_stride, h_stride,
                                         w_used//w_stride, w_stride).mean(axis=(1,3))

def pick_indices(N, n_samples, random_pick=True, seed=0, override=None):
    if override and len(override) > 0:
        return np.array([i for i in override if 0 <= i < N], dtype=int)
    n = min(n_samples, N)
    if random_pick:
        rng = np.random.default_rng(seed)
        return np.sort(rng.choice(N, size=n, replace=False))
    else:
        return np.arange(n, dtype=int)

def safe_decode(x):
    try:
        return x.decode() if isinstance(x, (bytes, bytearray, np.bytes_)) else str(x)
    except Exception:
        return str(x)

# --------- Load (lazy; don’t pull all data into RAM) ---------
with h5py.File(h5_path, "r") as f:
    if "data" not in f:
        raise KeyError("Dataset 'data' not found in the H5 file.")

    data = f["data"]              # (N, H, W) — HDF5 dataset, not loaded
    N = data.shape[0]
    print(f"[INFO] Found data with shape: {data.shape}")

    ts_ds = f.get("timestamps", None)  # optional
    fr_ds = f.get("freq_range", None)  # optional

    # Decide which indices to show
    idxs = pick_indices(N, n_samples, random_pick=random_pick, seed=seed, override=indices_override)
    print(f"[INFO] Visualizing {len(idxs)} samples: {idxs[:10]}{' ...' if len(idxs)>10 else ''}")

    # Grid layout
    n_imgs = len(idxs)
    rows = int(np.ceil(n_imgs / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2*cols, 2*rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, idx in zip(axes, idxs):
        # Read one image from disk
        img = np.asarray(data[idx], dtype=np.float32)
        small = downsample(img, target_size)

        # Contrast (optional)
        if use_percentile_contrast:
            vmin, vmax = np.percentile(small, [1.0, 99.0])
        else:
            vmin, vmax = None, None

        ax.imshow(small, cmap=cmap, origin="lower", aspect="auto", vmin=vmin, vmax=vmax)

        # Title: idx + timestamp (if available)
        title = f"idx={idx}"
        if ts_ds is not None:
            try:
                title += f"\n{safe_decode(ts_ds[idx])}"
            except Exception:
                pass
        ax.set_title(title, fontsize=8)
        ax.axis("off")

    # Hide any unused axes
    for ax in axes[n_imgs:]:
        ax.axis("off")

    # Add a shared y-label with freq range if available
    if fr_ds is not None and len(idxs) > 0:
        try:
            fr = np.asarray(fr_ds[idxs[0]], dtype=np.float32)
            fig.text(0.005, 0.5, f"freq ({fr[0]:.2f}–{fr[1]:.2f} MHz)", va="center", rotation=90, fontsize=9)
        except Exception:
            pass

    plt.tight_layout()
    plt.show()


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '../../../../project/lofarsw/Data/Dynspec/unlabelled_dataset_oriented_unf.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)